<a href="https://colab.research.google.com/github/valeriebarajasortiz-dev/examen-3/blob/main/NOVA_Examen_Informatica_Medica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NOVA · Mi primer chatbot de Informática Médica
**Grupo 13602-A · Práctica en Google Colab**

En equipos, van a escribir preguntas sobre Informática Médica, asignarles una **etiqueta** (el tema de la pregunta), subirlas a GitHub y diseñar el aspecto del chatbot. Primero entrenarán el Transformer con sus etiquetas. Luego podrán escribir una pregunta completa o unas pocas palabras y ver la respuesta que el equipo preparó para el tema detectado.

Usen ejemplos inventados. Eviten nombres, documentos, historias clínicas y otros datos reales, tanto en GitHub como en el chat. NOVA es un prototipo educativo de informática en salud. Las consultas sobre síntomas, diagnósticos y tratamientos quedan fuera de la práctica.

Descarga el archivo `etiquetas_medicas_completo.csv` que acompaña este cuaderno. En Colab, abre **Archivo → Subir cuaderno** y elige el archivo `.ipynb`. Ejecuta las celdas con ▶ de arriba hacia abajo. Si dejan vacío el enlace, NOVA usa las mismas 150 preguntas del CSV incluido: 15 por cada uno de los 10 temas. Para trabajar con su versión, peguen el enlace Raw de su repositorio.

## 1. Suban sus etiquetas a GitHub

1. Abran el CSV de ejemplo en su hoja de cálculo. Cada fila tiene una pregunta en `texto` y una etiqueta en `categoria`. Ejemplo: «¿Qué es la telemedicina?» → `telemedicina`.
2. Escriban preguntas distintas para al menos dos temas, además de `fuera_de_tema`. Usen etiquetas cortas, sin espacios ni tildes. Cada etiqueta necesita al menos tres preguntas diferentes para que el cuaderno pueda separar entrenamiento, validación y prueba. El archivo incluye 15 ejemplos por etiqueta. Procuren mantener cantidades similares entre temas.
3. Guarden el archivo como **CSV UTF-8**. En un repositorio público de práctica de su equipo, elijan **Add file → Upload files**, suban el CSV y confirmen con **Commit changes**.
4. Abran el archivo en GitHub y copien el enlace de **Raw**. Péguenlo abajo entre las comillas. El enlace debe comenzar con `https://raw.githubusercontent.com/`. La celda mostrará cuántas preguntas hay por etiqueta.

In [1]:
# @title Cargar etiquetas desde GitHub
URL_CSV = "https://raw.githubusercontent.com/valeriebarajasortiz-dev/examen-3/refs/heads/main/etiquetas_medicas%20(1).csv" # @param {type:"string"}

import re
from urllib.parse import urlsplit
import pandas as pd
from IPython.display import display

EJEMPLOS = {
    "historia_clinica": [
        "¿Qué diferencia hay entre una historia clínica electrónica y una carpeta escaneada?",
        "¿En qué sección del registro clínico suelen quedar los antecedentes?",
        "¿Para qué se identifica al autor de una nota en la historia?",
        "¿Cómo distingo la fecha de atención de la fecha en que se registró la nota?",
        "¿Qué información organiza una hoja de evolución?",
        "¿Qué datos quedan documentados al cerrar un episodio de atención?",
        "¿Un resumen de egreso forma parte de la historia electrónica?",
        "¿Por qué el sistema conserva versiones de una nota corregida?",
        "¿Qué diferencia hay entre un registro de urgencias y la historia longitudinal?",
        "¿Dónde se consulta el historial de atenciones anteriores?",
        "¿Qué apartados tendría una historia clínica ficticia para esta práctica?",
        "¿Qué representa una nota firmada dentro del expediente electrónico?",
        "¿Cómo se documenta que una nota se agregó después de la atención?",
        "¿Para qué sirve agrupar notas y resultados por episodio?",
        "¿Qué diferencia hay entre una nota narrativa y un campo estructurado?"
    ],
    "privacidad": [
        "¿Puedo poner el nombre de un paciente real en el CSV del equipo?",
        "¿Una captura con el número de historia visible contiene datos identificables?",
        "¿Qué significa usar solo los datos necesarios para una actividad de clase?",
        "¿Quitar el nombre de una tabla basta si todavía aparecen fechas y lugares específicos?",
        "¿Qué diferencia hay entre inventar registros y copiar registros reales?",
        "¿Por qué no conviene publicar historias reales en un repositorio público?",
        "¿Qué reviso antes de compartir una captura del chat de práctica?",
        "¿Un correo electrónico puede identificar a alguien en una base de salud?",
        "¿Qué quiere decir limitar el uso de los datos al propósito informado?",
        "¿De qué forma una combinación de edad y municipio puede identificar a una persona?",
        "¿Qué es seudonimizar un conjunto de datos?",
        "¿Un identificador reemplazado por un código todavía puede vincularse con alguien?",
        "¿Por qué los nombres de archivo también pueden revelar información personal?",
        "¿Qué significa confidencialidad al revisar información de salud?",
        "¿Puedo trabajar con un caso completamente inventado para demostrar la interfaz?"
    ],
    "seguridad_y_accesos": [
        "¿Qué significa que el acceso a la historia clínica dependa del rol?",
        "¿Para qué sirve la autenticación de dos factores en una aplicación clínica?",
        "¿Deberían dos estudiantes usar la misma cuenta para entrar al sistema?",
        "¿Qué registra una bitácora de acceso a un expediente?",
        "¿Cómo se sabe quién consultó una ficha y a qué hora?",
        "¿Por qué un perfil de lectura no puede editar registros?",
        "¿Qué se hace con los permisos cuando alguien cambia de función?",
        "¿Para qué sirve que el sistema cierre una sesión inactiva?",
        "¿Qué diferencia hay entre iniciar sesión y tener permiso para ver una nota?",
        "¿Por qué se recomienda bloquear el equipo al salir del consultorio?",
        "¿Qué significa asignar privilegios mínimos a un usuario?",
        "¿Qué problema causa una cuenta compartida para la auditoría?",
        "¿Qué tipo de evento puede registrar un sistema al modificar un permiso?",
        "¿Para qué se revisan periódicamente las cuentas habilitadas?",
        "¿Cómo se relacionan las copias de respaldo con recuperar un sistema?"
    ],
    "telemedicina": [
        "¿Qué diferencia hay entre una consulta por video y una consulta asincrónica?",
        "¿Para qué sirve verificar el micrófono antes de una consulta remota?",
        "¿Qué significa seguimiento remoto dentro de la telesalud?",
        "¿Una videollamada clínica es un ejemplo de interacción sincrónica?",
        "¿Cómo se documenta que una atención se realizó a distancia?",
        "¿Qué datos de agenda necesita una sesión de telemedicina ficticia?",
        "¿Para qué sirve una sala de espera virtual?",
        "¿Qué diferencia hay entre teleconsulta y teleorientación en una simulación?",
        "¿Qué pasa con la modalidad de atención si la videollamada se interrumpe?",
        "¿Cómo se distingue un mensaje enviado para revisión posterior de una sesión en vivo?",
        "¿Por qué el canal de comunicación importa en una consulta remota?",
        "¿Qué dificultades de conectividad pueden afectar una demostración de teleconsulta?",
        "¿Para qué se comprueba el audio y la cámara en una cita virtual?",
        "¿Qué es el monitoreo remoto en términos de informática en salud?",
        "¿Cómo identificar en la agenda que una cita se hará por videollamada?"
    ],
    "interoperabilidad": [
        "¿Cómo puede un sistema de laboratorio enviar resultados a otro sistema clínico?",
        "¿Qué significa que dos aplicaciones entiendan igual un dato que intercambian?",
        "¿Para qué sirve el estándar HL7 FHIR?",
        "¿Qué representa un recurso Observation en FHIR?",
        "¿Qué diferencia hay entre un recurso Patient y uno Encounter?",
        "¿Por qué dos hospitales necesitan acordar el formato de los mensajes?",
        "¿Cómo se conserva el significado de un dato cuando pasa de un sistema a otro?",
        "¿Qué es la interoperabilidad semántica?",
        "¿Enviar un PDF y enviar datos estructurados son la misma forma de intercambio?",
        "¿Por qué es importante acordar unidades y códigos al intercambiar resultados?",
        "¿Qué ventaja tiene usar recursos estándar en una integración?",
        "¿Cómo se representa una visita cuando se usa un recurso Encounter?",
        "¿Para qué sirve una interfaz que conecta farmacia y registro electrónico?",
        "¿Qué significa recibir información que la aplicación puede interpretar automáticamente?",
        "¿Puede haber intercambio de archivos sin que ambos sistemas entiendan su contenido?"
    ],
    "terminologias_clinicas": [
        "¿Para qué sirve un código CIE-10 en un registro de práctica?",
        "¿Qué diferencia hay entre la descripción escrita y el código de un concepto?",
        "¿Qué tipo de conceptos representa SNOMED CT?",
        "¿Qué catálogo se usa para identificar una observación de laboratorio?",
        "¿Por qué dos sinónimos pueden apuntar a un mismo concepto?",
        "¿CIE-10 y SNOMED CT tienen el mismo propósito?",
        "¿LOINC identifica la prueba o el valor obtenido?",
        "¿Qué significa mapear un concepto entre dos sistemas de codificación?",
        "¿Cómo ayuda una terminología controlada a buscar registros?",
        "¿Por qué es importante indicar a qué catálogo pertenece un código?",
        "¿Qué es un sinónimo dentro de una terminología clínica?",
        "¿Qué pasa si dos servicios usan códigos locales distintos para la misma prueba?",
        "¿Cómo se distingue el nombre visible del identificador de un concepto?",
        "¿Para qué se usa un catálogo normalizado en un formulario clínico?",
        "¿Qué ventaja tiene registrar un código junto con el texto libre?"
    ],
    "calidad_de_datos": [
        "¿Cómo detecto filas duplicadas en un conjunto de datos ficticio?",
        "¿Qué diferencia hay entre un dato faltante y un valor de cero?",
        "¿Por qué dos formatos de fecha pueden causar errores en una tabla?",
        "¿Qué significa que un campo obligatorio esté incompleto?",
        "¿Cómo noto que una fecha de egreso es anterior al ingreso?",
        "¿Qué diferencia hay entre completitud y consistencia de los datos?",
        "¿Por qué conviene registrar las unidades en una columna definida?",
        "¿Qué reviso si dos campos del mismo registro se contradicen?",
        "¿Cómo afectan los espacios sobrantes a una comparación de etiquetas?",
        "¿Qué significa que una columna tenga valores fuera de su rango esperado?",
        "¿Para qué se documenta la corrección de una fila?",
        "¿Cómo distingo un registro duplicado de dos atenciones diferentes?",
        "¿Qué problemas puede generar mezclar números y texto en una misma columna?",
        "¿Por qué es importante que todos usen la misma convención para datos desconocidos?",
        "¿Qué validación ayuda a encontrar identificadores vacíos en un archivo?"
    ],
    "soporte_decisiones": [
        "¿Qué es un sistema de apoyo a la decisión clínica?",
        "¿Para qué sirve una alerta de seguridad dentro de una aplicación clínica?",
        "¿Qué significa fatiga por alertas?",
        "¿Cómo se distingue un recordatorio automático de una nota escrita por alguien?",
        "¿Qué información muestra una alerta para explicar por qué apareció?",
        "¿Por qué es útil registrar si una alerta se aceptó o se descartó?",
        "¿Qué diferencia hay entre una regla del sistema y una decisión humana?",
        "¿Qué es un falso positivo en una alerta informática?",
        "¿Cómo evaluamos si una regla simulada se activa cuando corresponde?",
        "¿Por qué demasiados avisos pueden hacer que se pasen por alto otros?",
        "¿Para qué sirve probar una alerta con casos ficticios antes de usarla?",
        "¿Qué significa que una alerta se active por un campo del registro?",
        "¿Cómo se documenta la versión de una regla en un sistema?",
        "¿Qué diferencia hay entre una sugerencia informativa y un aviso que interrumpe el flujo?",
        "¿Por qué se revisa la pertinencia de las alertas después de implementarlas?"
    ],
    "evaluacion_chatbot": [
        "¿Qué muestra una matriz de confusión del chatbot?",
        "¿Qué diferencia hay entre preguntas de entrenamiento y de prueba?",
        "¿Por qué el porcentaje de aciertos puede ocultar fallos en una etiqueta poco frecuente?",
        "¿Cómo sé si el chatbot confunde privacidad con seguridad y accesos?",
        "¿Qué significa que el modelo tenga un puntaje alto y se equivoque?",
        "¿Para qué se guardan preguntas que el modelo no vio durante el entrenamiento?",
        "¿Qué es una clase desbalanceada en el CSV?",
        "¿Por qué no conviene repetir la misma pregunta en entrenamiento y prueba?",
        "¿Cómo se interpreta un error de clasificación en el prototipo?",
        "¿Qué significa sobreajuste con pocos ejemplos?",
        "¿Qué puedo revisar si el chat predice siempre el mismo tema?",
        "¿Por qué usar frases nuevas para evaluar el chatbot?",
        "¿Qué diferencia hay entre acertar una etiqueta y tener una respuesta bien redactada?",
        "¿Qué indica el F1 macro al comparar categorías?",
        "¿Cómo registro un caso de prueba donde la etiqueta esperada y la predicha son distintas?"
    ],
    "fuera_de_tema": [
        "¿Puedes recomendarme una película para el fin de semana?",
        "¿Cuántos goles lleva mi equipo favorito?",
        "¿Cómo cambio la rueda de una bicicleta?",
        "¿Qué tiempo va a hacer mañana en Bucaramanga?",
        "¿Puedes resolver esta ecuación de segundo grado?",
        "¿Qué me recomiendas tomar para un dolor de cabeza?",
        "¿Puedes diagnosticarme según los síntomas que te describa?",
        "¿Qué significa que me duela el pecho?",
        "¿Me puedes interpretar mi resultado de laboratorio personal?",
        "¿Qué tratamiento necesito para una infección?",
        "¿Puedes hacerme un plan para correr una maratón?",
        "¿Cómo preparo una arepa?",
        "¿Cuál es el precio del dólar hoy?",
        "¿Qué dosis debo usar de mi medicamento?",
        "¿Me cuentas un chiste sobre pingüinos?"
    ]
}

def normalizar(texto):
    return " ".join(texto.strip().casefold().split())

def cargar_etiquetas(url):
    if url.strip():
        enlace = urlsplit(url.strip())
        if enlace.scheme != "https" or enlace.netloc != "raw.githubusercontent.com":
            raise ValueError("Abre el CSV en GitHub, pulsa Raw y copia ese enlace.")
        try:
            datos = pd.read_csv(url.strip(), encoding="utf-8-sig", sep=None, engine="python",
                                dtype=str, keep_default_na=False)
        except (OSError, UnicodeError, pd.errors.ParserError, pd.errors.EmptyDataError) as error:
            raise ValueError("No se pudo leer el CSV. Revisa que sea público, esté guardado en UTF-8 y que el enlace Raw abra el archivo.") from error
    else:
        datos = pd.DataFrame([(texto, tema) for tema, textos in EJEMPLOS.items()
                             for texto in textos], columns=["texto", "categoria"])
    datos.columns = datos.columns.str.strip().str.lower()
    if not {"texto", "categoria"}.issubset(datos.columns):
        raise ValueError("La primera fila del CSV debe contener texto y categoria.")
    datos = datos[["texto", "categoria"]].copy()
    datos["texto"] = datos["texto"].str.strip()
    datos["categoria"] = datos["categoria"].str.strip()
    if (datos["texto"] == "").any() or not datos["categoria"].str.fullmatch(r"[a-z][a-z0-9_]*").all():
        raise ValueError("Completa todas las filas. Usa etiquetas como historia_clinica, en minúsculas y sin tildes.")
    clave = datos["texto"].map(normalizar)
    if (datos.groupby(clave)["categoria"].nunique() > 1).any():
        raise ValueError("Hay una pregunta repetida con dos etiquetas distintas. Revisa el CSV.")
    datos = datos.loc[~clave.duplicated()].reset_index(drop=True)
    conteos = datos["categoria"].value_counts()
    if "fuera_de_tema" not in conteos or len(conteos) < 3 or conteos.min() < 3:
        raise ValueError("Incluye dos temas y fuera_de_tema, con al menos tres preguntas distintas por etiqueta.")
    return datos

predictor_transformer = None
df = cargar_etiquetas(URL_CSV)
labels = sorted(df["categoria"].unique())
print("Origen:", "GitHub" if URL_CSV.strip() else "Ejemplos del cuaderno")
display(df["categoria"].value_counts().rename("Preguntas").to_frame())
display(df.head())

Origen: GitHub


,Preguntas
categoria,
historia_clinica,15
privacidad,15
seguridad_y_accesos,15
telemedicina,15
interoperabilidad,15
terminologias_clinicas,15
calidad_de_datos,15
soporte_decisiones,15
evaluacion_chatbot,15


,texto,categoria
0,¿Qué diferencia hay entre una historia clínica...,historia_clinica
1,¿En qué sección del registro clínico suelen qu...,historia_clinica
2,¿Para qué se identifica al autor de una nota e...,historia_clinica
3,¿Cómo distingo la fecha de atención de la fech...,historia_clinica
4,¿Qué información organiza una hoja de evolución?,historia_clinica


## 2. Entrenen y evalúen el Transformer

Ejecuten las tres celdas. Para acelerar el entrenamiento, pueden elegir una GPU en **Entorno de ejecución → Cambiar tipo de entorno de ejecución** antes de comenzar. Si vienen de una versión anterior, reinicien la sesión de Colab y ejecuten desde el principio.

Con el CSV incluido hay 90 preguntas para entrenar, 30 para validar y 30 para probar. Revisen la columna **Predijo** y la matriz de confusión para identificar los temas que el Transformer confunde. Esta evaluación mide el Transformer por sí solo. En el paso 3 podrán añadir palabras clave para consultas breves.

La primera descarga de los pesos puede tardar varios minutos.

In [32]:
# Configura la descarga antes de importar Hugging Face.
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

%pip -q install "transformers==4.57.3" "huggingface_hub==0.36.0" "datasets>=3,<5" "accelerate>=1,<2" scikit-learn

In [33]:
# @title Entrenar con el CSV cargado
import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score
SEED = 42
num_labels = len(labels)
ids = {etiqueta: i for i, etiqueta in enumerate(labels)}
datos = df.assign(label=df["categoria"].map(ids))
partes_train, partes_val, partes_test = [], [], []
for _, grupo in datos.groupby("label"):
    mezclado = grupo.sample(frac=1, random_state=SEED)
    reserva = max(1, len(mezclado) // 5)
    partes_val.append(mezclado.iloc[:reserva])
    partes_test.append(mezclado.iloc[reserva:2 * reserva])
    partes_train.append(mezclado.iloc[2 * reserva:])
train_df = pd.concat(partes_train, ignore_index=True)
val_df = pd.concat(partes_val, ignore_index=True)
test_df = pd.concat(partes_test, ignore_index=True)
print(f"Entrenamiento: {len(train_df)} · Validación: {len(val_df)} · Prueba: {len(test_df)}")

from pathlib import Path

from datasets import Dataset
from transformers import DistilBertTokenizerFast, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
from huggingface_hub import hf_hub_download

MODELO_BASE = "distilbert/distilbert-base-multilingual-cased"
print('Descargando tokenizer y preparando datos...')
# DistilBERT clasifica textos. Carga el tokenizador desde su archivo oficial.
archivo_tokenizador = hf_hub_download(repo_id=MODELO_BASE, filename="tokenizer.json")
tokenizer = DistilBertTokenizerFast(tokenizer_file=archivo_tokenizador, do_lower_case=False)
def preparar(tabla):
    datos=Dataset.from_pandas(tabla[['texto','label']].reset_index(drop=True),preserve_index=False)
    return datos.map(lambda lote:tokenizer(lote['texto'],truncation=True,max_length=96),batched=True)
train_ds=preparar(train_df); val_ds=preparar(val_df); test_ds=preparar(test_df)
collator=DataCollatorWithPadding(tokenizer=tokenizer)
print("Descargando los pesos de DistilBERT. La primera vez puede tardar varios minutos...")
hf_hub_download(repo_id=MODELO_BASE, filename="config.json")
archivo_pesos = hf_hub_download(repo_id=MODELO_BASE, filename="model.safetensors")
carpeta_modelo = str(Path(archivo_pesos).parent)

def crear_modelo():
    return AutoModelForSequenceClassification.from_pretrained(
        carpeta_modelo, num_labels=num_labels, use_safetensors=True, local_files_only=True,
        id2label={i:str(v) for i,v in enumerate(labels)},
        label2id={str(v):i for i,v in enumerate(labels)})
def calcular_metricas(pred):
    estimadas=np.argmax(pred.predictions,axis=1)
    return {'accuracy':accuracy_score(pred.label_ids,estimadas),
            'f1_macro':f1_score(pred.label_ids,estimadas,average='macro',zero_division=0)}
print('PASO 5 COMPLETADO:',len(train_ds),'ejemplos de entrenamiento tokenizados.')

args = TrainingArguments(
    output_dir="/content/nova_medica", num_train_epochs=10, learning_rate=5e-5,
    per_device_train_batch_size=4, per_device_eval_batch_size=4,
    eval_strategy="epoch", save_strategy="no", report_to="none", seed=SEED,
)
torch.manual_seed(SEED)
trainer = Trainer(model=crear_modelo(), args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  processing_class=tokenizer, data_collator=collator, compute_metrics=calcular_metricas)
trainer.train()
modelo = trainer.model
modelo.eval()

def clasificar(texto):
    entradas = tokenizer(texto, return_tensors="pt", truncation=True, max_length=96).to(modelo.device)
    with torch.inference_mode():
        puntajes = modelo(**entradas).logits[0].softmax(dim=-1)
    mejor = int(puntajes.argmax().item())
    return labels[mejor], float(puntajes[mejor].item())

predictor_transformer = clasificar
print("Entrenamiento completo. Continúa con la evaluación y el diseño del chat.")

Entrenamiento: 90 · Validación: 30 · Prueba: 30
Descargando tokenizer y preparando datos...


Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Descargando los pesos de DistilBERT. La primera vez puede tardar varios minutos...
PASO 5 COMPLETADO: 90 ejemplos de entrenamiento tokenizados.


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at /root/.cache/huggingface/hub/models--distilbert--distilbert-base-multilingual-cased/snapshots/45c032ab32cc946ad88a166f7cb282f58c753c2e and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,2.301585,0.100000,0.018182
2,No log,2.254518,0.200000,0.093864
3,No log,2.112580,0.233333,0.138571
4,No log,2.021528,0.233333,0.232381
5,No log,1.930480,0.333333,0.377738
6,No log,1.862042,0.400000,0.417051
7,No log,1.850077,0.300000,0.316667
8,No log,1.845675,0.333333,0.383333
9,No log,1.771796,0.400000,0.378333
10,No log,1.791074,0.300000,0.318095


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argume

Entrenamiento completo. Continúa con la evaluación y el diseño del chat.


In [34]:
# @title Revisar las preguntas reservadas
salida = trainer.predict(test_ds)
revision = test_df[["texto", "categoria"]].rename(columns={"texto": "Pregunta", "categoria": "Etiqueta esperada"})
revision["Predijo"] = [labels[i] for i in np.argmax(salida.predictions, axis=1)]
display(revision)
print(f"Aciertos: {(revision['Etiqueta esperada'] == revision['Predijo']).sum()} de {len(revision)}")

# Filas: tema esperado. Columnas: tema que eligió el modelo.
matriz = pd.crosstab(revision["Etiqueta esperada"], revision["Predijo"])
display(matriz.reindex(index=labels, columns=labels, fill_value=0))
f1 = f1_score(revision["Etiqueta esperada"], revision["Predijo"], average="macro", zero_division=0)
print(f"F1 macro: {f1:.2f} (promedia el desempeño de los temas con el mismo peso).")


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


,Pregunta,Etiqueta esperada,Predijo
0,¿Por qué es importante que todos usen la misma...,calidad_de_datos,calidad_de_datos
1,¿Qué diferencia hay entre completitud y consis...,calidad_de_datos,privacidad
2,¿Cómo afectan los espacios sobrantes a una com...,calidad_de_datos,evaluacion_chatbot
3,¿Qué indica el F1 macro al comparar categorías?,evaluacion_chatbot,terminologias_clinicas
4,¿Para qué se guardan preguntas que el modelo n...,evaluacion_chatbot,evaluacion_chatbot
5,¿Cómo se interpreta un error de clasificación ...,evaluacion_chatbot,soporte_decisiones
6,¿Qué dosis debo usar de mi medicamento?,fuera_de_tema,evaluacion_chatbot
7,¿Qué me recomiendas tomar para un dolor de cab...,fuera_de_tema,fuera_de_tema
8,¿Me puedes interpretar mi resultado de laborat...,fuera_de_tema,fuera_de_tema
9,¿Para qué sirve agrupar notas y resultados por...,historia_clinica,privacidad


Aciertos: 14 de 30


Predijo,calidad_de_datos,evaluacion_chatbot,fuera_de_tema,historia_clinica,interoperabilidad,privacidad,seguridad_y_accesos,soporte_decisiones,telemedicina,terminologias_clinicas
Etiqueta esperada,,,,,,,,,,
calidad_de_datos,1,1,0,0,0,1,0,0,0,0
evaluacion_chatbot,0,1,0,0,0,0,0,1,0,1
fuera_de_tema,0,1,2,0,0,0,0,0,0,0
historia_clinica,0,0,0,1,0,2,0,0,0,0
interoperabilidad,1,0,0,0,1,1,0,0,0,0
privacidad,0,0,0,0,0,3,0,0,0,0
seguridad_y_accesos,1,0,0,0,0,2,0,0,0,0
soporte_decisiones,0,1,0,0,1,0,0,1,0,0
telemedicina,0,0,0,0,1,0,0,0,2,0


F1 macro: 0.46 (promedia el desempeño de los temas con el mismo peso).


## 3. Personalicen el chat, las respuestas y las palabras clave

En **Diseña el chatbot de tu equipo**, elijan la paleta, el icono, el nombre, la bienvenida y los textos de los botones. En la siguiente celda, editen la respuesta que se mostrará para cada tema.

En **Palabras clave por tema**, escriban palabras o frases cortas que identifiquen bien una etiqueta. Por ejemplo, `fhir` apunta a `interoperabilidad` y `consulta virtual` a `telemedicina`. El chat acepta mayúsculas y texto con o sin tildes. Si el mismo término apunta a varios temas, pide precisar. Si escriben una pregunta completa o el término no está en la lista, el Transformer intenta clasificarla.

Después de editar cualquiera de estas celdas, ejecútenla y vuelvan a abrir el chat. Los cambios de diseño, palabras clave y respuestas se aplican sin repetir el entrenamiento.

In [36]:
# @title Diseña el chatbot de tu equipo
NOMBRE = "Pepita " # @param {type:"string"}
EQUIPO = "Grupo 13602-A" # @param {type:"string"}
BIENVENIDA = "Bienvenido pon a prueba tus conocimientos en informática en salud con preguntas prácticas." # @param {type:"string"}
COLOR = "Violeta" # @param ["Verde","Azul","Violeta","Terracota","celeste "]
ICONO = "Salud" # @param ["Salud", "Conversación", "Libro"]
ALTURA_CHAT = 590 # @param {type:"slider", min:300, max:650, step:10}
ETIQUETA_CAMPO = "Tu pregunta" # @param {type:"string"}
EJEMPLO = "¿Qué diferencia hay entre una historia clínica electrónica y una carpeta escaneada?" # @param {type:"string"}
TEXTO_BOTON = "Enviar pregunta" # @param {type:"string"}
TEXTO_LIMPIAR = "Nueva conversación" # @param {type:"string"}
MOSTRAR_EJEMPLOS = True # @param {type:"boolean"}
MOSTRAR_MODO = True # @param {type:"boolean"}

In [38]:
# Añade una línea por cada etiqueta de tu equipo, con este formato:
# "mi_etiqueta": "Respuesta educativa del equipo.",
RESPUESTAS = {
    "historia_clinica": "La historia clínica electrónica reúne la información registrada durante la atención. Sus secciones organizan antecedentes, notas, resultados y otros registros con fechas y responsables.",
    "privacidad": "La privacidad se relaciona con qué información personal se usa y con qué propósito. Para esta práctica, trabajen con datos ficticios y revisen que los archivos y capturas carezcan de identificadores.",
    "seguridad_y_accesos": "Los permisos por rol limitan qué funciones puede usar cada persona. Las cuentas individuales, la autenticación y la auditoría ayudan a proteger el sistema y a registrar el uso.",
    "telemedicina": "La telemedicina usa tecnologías de comunicación para prestar servicios de salud a distancia. Una interacción puede ocurrir en tiempo real o mediante información que se revisa después.",
    "interoperabilidad": "La interoperabilidad permite que distintos sistemas intercambien datos y los interpreten de forma consistente. FHIR organiza el intercambio mediante recursos como Patient, Encounter y Observation.",
    "terminologias_clinicas": "Las terminologías y clasificaciones ayudan a representar conceptos con códigos consistentes. CIE-10 clasifica enfermedades y otros problemas de salud, SNOMED CT representa conceptos clínicos y LOINC identifica observaciones y pruebas.",
    "calidad_de_datos": "La calidad de datos se revisa comprobando si los registros son completos, válidos, coherentes y consistentes. Conviene documentar las correcciones y distinguir los datos faltantes de los valores válidos.",
    "soporte_decisiones": "Un sistema de apoyo a decisiones puede mostrar alertas o recordatorios basados en reglas o datos. En esta práctica se analiza cómo funciona la herramienta y cómo se revisan sus resultados.",
    "evaluacion_chatbot": "Un clasificador se evalúa con preguntas que quedaron reservadas. Comparen la etiqueta esperada con la predicción y revisen los errores por tema."
}
FUERA_DE_TEMA = "Para esta práctica, haz una pregunta sobre informática en salud o sobre el funcionamiento del chatbot. Las consultas clínicas individuales requieren atención profesional."

faltantes = sorted(set(labels) - set(RESPUESTAS) - {"fuera_de_tema"})
print("Agrega respuestas para:", ", ".join(faltantes)) if faltantes else print("Las etiquetas tienen respuestas.")

Las etiquetas tienen respuestas.


In [40]:
# @title Palabras clave por tema
# Copia cada etiqueta tal como aparece en el CSV. Edita los términos entre comillas.
PALABRAS_CLAVE = {
    "historia_clinica": ["historia clínica", "expediente clínico", "nota de evolución", "egreso"],
    "privacidad": ["privacidad", "confidencialidad", "anonimización", "seudonimización", "datos personales"],
    "seguridad_y_accesos": ["seguridad", "accesos", "permisos", "autenticación", "auditoría", "contraseña"],
    "telemedicina": ["telemedicina", "telesalud", "teleconsulta", "consulta virtual", "videollamada"],
    "interoperabilidad": ["interoperabilidad", "fhir", "hl7", "observation", "intercambio de datos"],
    "terminologias_clinicas": ["terminología", "cie-10", "cie 10", "snomed", "snomed ct", "loinc"],
    "calidad_de_datos": ["calidad de datos", "duplicados", "datos faltantes", "completitud", "consistencia"],
    "soporte_decisiones": ["alertas", "fatiga por alertas", "recordatorios", "apoyo a decisiones"],
    "evaluacion_chatbot": ["matriz de confusión", "sobreajuste", "f1", "f1 macro", "clasificación"],
    "fuera_de_tema": ["diagnóstico", "tratamiento", "dosis", "síntomas", "receta"],
}

import unicodedata
from collections import defaultdict

def palabras(texto):
    texto = unicodedata.normalize("NFD", texto.casefold())
    sin_tildes = "".join(c for c in texto if not unicodedata.combining(c))
    return tuple(re.findall(r"[a-z0-9]+", sin_tildes))

indice_claves = defaultdict(set)
for etiqueta, terminos in PALABRAS_CLAVE.items():
    if etiqueta in labels:
        for termino in terminos:
            clave = palabras(termino)
            if clave:
                indice_claves[clave].add(etiqueta)

claves_ambiguas = [" ".join(clave) for clave, temas in indice_claves.items() if len(temas) > 1]
if claves_ambiguas:
    print("Estas palabras apuntan a varios temas:", ", ".join(sorted(claves_ambiguas)))

def resolver_consulta(pregunta):
    tokens = palabras(pregunta)
    if not tokens:
        return "Escribe una palabra clave o una pregunta para continuar.", None
    if predictor_transformer is None:
        return "Ejecuta el paso 2 para entrenar el Transformer y luego abre el chat.", None

    # Para entradas cortas, prioriza una coincidencia completa o las frases más largas.
    if len(tokens) <= 5:
        coincidencias = indice_claves.get(tokens, set())
        if not coincidencias:
            encontradas = [(len(clave), temas) for clave, temas in indice_claves.items()
                           if any(tokens[i:i+len(clave)] == clave for i in range(len(tokens) - len(clave) + 1))]
            if encontradas:
                longitud = max(tamano for tamano, _ in encontradas)
                coincidencias = set().union(*(temas for tamano, temas in encontradas if tamano == longitud))
        if len(coincidencias) > 1:
            nombres = ", ".join(tema.replace("_", " ") for tema in sorted(coincidencias))
            return "Encontré varios temas: " + nombres + ". ¿Sobre cuál quieres preguntar?", None
    else:
        coincidencias = set()

    if coincidencias:
        etiqueta = next(iter(coincidencias))
        metodo = "Palabra clave"
    else:
        etiqueta, puntaje = predictor_transformer(pregunta)
        if puntaje < 0.55:
            return "No pude identificar el tema. Prueba con una palabra clave del panel o escribe una pregunta más concreta.", None
        metodo = "Transformer"

    texto = (FUERA_DE_TEMA if etiqueta == "fuera_de_tema" else
             RESPUESTAS.get(etiqueta, "El equipo todavía debe escribir la respuesta para " + etiqueta + "."))
    return texto, f"{metodo} · {etiqueta.replace('_', ' ')}"

def responder(pregunta):
    return resolver_consulta(pregunta)[0]

## 4. Abran el chatbot y prueben consultas cortas

Ejecuten las dos celdas para abrir la interfaz. En el panel lateral pueden consultar las palabras clave del tema elegido o cargar una pregunta del CSV. Prueben **FHIR**, **consulta virtual** y una pregunta formulada con sus propias palabras. Debajo de la conversación se indica cómo se reconoció el tema. Si el chat detecta varios temas, escriban uno de ellos con más precisión.

El enlace de Gradio es público y temporal. Usen ejemplos ficticios. Si el enlace falla, usen la prueba de texto.

In [24]:
%pip -q install "gradio==5.49.1" "huggingface_hub==0.36.0"

In [41]:
# @title Abrir o actualizar el chat
import gradio as gr
from html import escape

if predictor_transformer is None:
    raise RuntimeError("Completa el entrenamiento del paso 2 antes de abrir el chat.")

if globals().get("chat_nova") is not None:
    chat_nova.close()

# Cada opción incluye el color de Gradio y los dos tonos del encabezado.
paletas = {
    "Verde": ("teal", "#124B48", "#1F655B"),
    "Azul": ("blue", "#153759", "#245D85"),
    "Violeta": ("violet", "#402666", "#60468C"),
    "Terracota": ("orange", "#702D26", "#913D30"),
}
iconos = {"Salud": "✚", "Conversación": "☏", "Libro": "▤"}
tono, fondo, fondo_final = paletas[COLOR]
modo = "Transformer entrenado · palabras clave activas"
estado_inicial = "Escribe una pregunta o selecciona un ejemplo." if MOSTRAR_EJEMPLOS else "Escribe una pregunta para comenzar."
aviso = "Usa preguntas ficticias. Evita ingresar datos personales o clínicos reales."
css = """
.gradio-container { max-width: 1120px !important; margin: auto !important; }
#nova-hero { border-radius: 20px; padding: clamp(18px, 4vw, 34px); margin-bottom: 12px; }
#nova-hero h1 { color: white; margin: 6px 0; font-size: clamp(24px, 4vw, 36px); line-height: 1.15; }
#nova-hero p { color: white; margin: 5px 0; line-height: 1.5; }
#nova-hero .nova-modo { display: inline-block; padding: 4px 10px; margin-top: 8px;
  border: 1px solid #ffffff90; border-radius: 999px; font-size: 0.85rem; }
#nova-panel { padding: 12px; border: 1px solid var(--border-color-primary); border-radius: 16px; }
#nova-estado { font-size: 0.9rem; }
"""
modo_html = f'<p class="nova-modo">{escape(modo)}</p>' if MOSTRAR_MODO else ""
equipo_html = f"<p>{escape(EQUIPO)}</p>" if EQUIPO.strip() else ""
encabezado = (
    f'<header id="nova-hero" style="background:linear-gradient(120deg,{fondo},{fondo_final})">'
    f'{equipo_html}<h1><span aria-hidden="true">{iconos[ICONO]}</span> {escape(NOMBRE)}</h1>'
    f'<p>{escape(BIENVENIDA)}</p>{modo_html}</header>'
)

def nombre_tema(etiqueta):
    return etiqueta.replace("_", " ").capitalize()

temas = [(nombre_tema(etiqueta), etiqueta) for etiqueta in labels]
tema_inicial = next((etiqueta for etiqueta in labels if etiqueta != "fuera_de_tema"), labels[0])

def ejemplos_del_tema(etiqueta):
    return df.loc[df["categoria"] == etiqueta, "texto"].head(4).tolist()

def mostrar_claves(etiqueta):
    claves = PALABRAS_CLAVE.get(etiqueta, [])
    texto = escape(", ".join(claves)) if claves else "Este tema todavía no tiene palabras clave."
    return f"<p><strong>Palabras clave</strong><br>{texto}</p>"

def cambiar_tema(etiqueta):
    opciones = ejemplos_del_tema(etiqueta)
    return gr.update(choices=opciones, value=None), mostrar_claves(etiqueta)

def cargar_ejemplo(seleccion):
    return seleccion or EJEMPLO

def enviar(pregunta, historial):
    pregunta = (pregunta or "").strip()
    historial = historial or []
    if not pregunta:
        return "", historial, "Escribe una pregunta para continuar."
    respuesta, origen = resolver_consulta(pregunta)
    nuevos = [{"role": "user", "content": pregunta},
              {"role": "assistant", "content": respuesta}]
    total = len(historial) // 2 + 1
    estado = "1 pregunta en esta conversación." if total == 1 else f"{total} preguntas en esta conversación."
    return "", historial + nuevos, f"{estado} **{origen}**" if origen else estado

def reiniciar():
    return "", [], estado_inicial

with gr.Blocks(title=NOMBRE, theme=gr.themes.Soft(primary_hue=tono), css=css) as chat_nova:
    gr.HTML(encabezado)
    gr.Markdown(aviso)
    with gr.Row(equal_height=False):
        with gr.Column(scale=3, min_width=320):
            conversacion = gr.Chatbot(type="messages", label="Conversación", height=ALTURA_CHAT)
            pregunta = gr.Textbox(label=ETIQUETA_CAMPO, placeholder=EJEMPLO, lines=2)
            with gr.Row():
                boton = gr.Button(TEXTO_BOTON, variant="primary")
                limpiar = gr.Button(TEXTO_LIMPIAR)
            estado = gr.Markdown(estado_inicial, elem_id="nova-estado")
        if MOSTRAR_EJEMPLOS:
            with gr.Column(scale=1, min_width=245, elem_id="nova-panel"):
                gr.Markdown("### Preguntas de práctica")
                tema = gr.Dropdown(choices=temas, value=tema_inicial, label="Tema", interactive=True)
                claves = gr.HTML(mostrar_claves(tema_inicial))
                ejemplo = gr.Dropdown(choices=ejemplos_del_tema(tema_inicial), label="Pregunta del CSV", interactive=True)
                cargar = gr.Button("Usar esta pregunta")
                sugerida = gr.Button("Usar ejemplo del equipo")
        salidas = [pregunta, conversacion, estado]
    boton.click(enviar, [pregunta, conversacion], salidas, queue=False)
    pregunta.submit(enviar, [pregunta, conversacion], salidas, queue=False)
    limpiar.click(reiniciar, outputs=salidas, queue=False)
    if MOSTRAR_EJEMPLOS:
        tema.change(cambiar_tema, inputs=tema, outputs=[ejemplo, claves], queue=False)
        cargar.click(cargar_ejemplo, inputs=ejemplo, outputs=pregunta, queue=False)
        sugerida.click(lambda: EJEMPLO, outputs=pregunta, queue=False)

chat_nova.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fd4de06980e23bafd3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [43]:
# @title Probar sin abrir la interfaz
CONSULTA = "FHIR" # @param {type:"string"}
respuesta, origen = resolver_consulta(CONSULTA)
print(respuesta)
if origen:
    print(origen)

La interoperabilidad permite que distintos sistemas intercambien datos y los interpreten de forma consistente. FHIR organiza el intercambio mediante recursos como Patient, Encounter y Observation.
Palabra clave · interoperabilidad


## 5. Entrega del equipo

- Enlace al CSV en GitHub y copia del cuaderno de Colab.
- Captura del chatbot con el nombre del equipo, la paleta y la bienvenida personalizados, y una pregunta de práctica enviada desde el panel.
- Prueba de tres preguntas inventadas: una palabra clave, una pregunta completa y una consulta fuera de tema. Anoten qué respondió y qué mejorarían en los ejemplos o las respuestas.

Si cambian preguntas o etiquetas en GitHub, ejecuten de nuevo la carga y el entrenamiento. Para cambiar solo el color, los textos de la interfaz o las respuestas escritas por el equipo, actualicen el chat sin repetir el entrenamiento.